# M3 · Ejecución integral del sistema RAG clínico

Une el trabajo de **todo el equipo** para el Módulo 3 y corre el pipeline de
punta a punta:

```
corpus (Isa)  →  retrieval + orquestación (Pau)  →  generación (Agustín)  →  RAGAS + cruce con M2 (Luis)
```

**Funciona igual en Google Colab y en local.** El notebook detecta el entorno y
se adapta: en Colab clona la rama `dev` y monta Drive; en local usa el repositorio
donde ya está y un directorio de datos propio.

Las celdas son Python puro (sin `%pip`, `!` ni `%cd`), así que corren en cualquier
kernel (Colab, Jupyter, VS Code) y también se pueden ejecutar como script.

**Requisitos:** `GROQ_API_KEY` (generación y RAGAS) y, opcionalmente,
`BIOPORTAL_API_KEY` (tool de normalización real). En Colab van en Secretos; en
local en el entorno o en un `.env`.

## 1 · Entorno y parámetros

Detecta Colab vs. local y define las rutas. En local se puede sobrescribir con
`M3_REPO_DIR` (raíz del repo) y `M3_DATA_ROOT` (datos), si no se toman los valores
por defecto.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


def _raiz_repo(inicio):
    p = Path(inicio).resolve()
    for cand in [p, *p.parents]:
        if (cand / "M3").is_dir() and (cand / ".git").exists():
            return cand
    return p


RAMA = "dev"
REPO_URL = "https://github.com/luisNP21/Topicos-IA.git"

if EN_COLAB:
    REPO_DIR = Path("/content/Topicos-IA")
    DATA_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3")
else:
    REPO_DIR = _raiz_repo(os.environ.get("M3_REPO_DIR") or os.getcwd())
    DATA_ROOT = Path(os.environ.get("M3_DATA_ROOT") or (REPO_DIR.parent / "m3_data")).resolve()

# Eval set del equipo (15 casos: entidad + pregunta + esperado).
EJEMPLOS = "M3/ejecucion/eval_set_casos.json"
EJEMPLOS_ALT = "M3/generacion/entidades_ejemplo.json"

MODO_RAGAS = "real"      # "real" (librería ragas + juez Groq) | "mock"
CORRER_CORPUS = True     # False si el índice ya está construido
CORRER_EXPERIMENTO = True  # calibra umbrales y mide el delta del retrieval

print("Entorno:", "Colab" if EN_COLAB else "local")
print("REPO_DIR:", REPO_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("RAGAS:", MODO_RAGAS, "| corpus:", CORRER_CORPUS, "| experimento:", CORRER_EXPERIMENTO)

## 2 · Sincronizar el repositorio

En **Colab** se clona/sincroniza `dev` (el runtime se pierde al cerrar la sesión).
En **local** se usa el repositorio tal cual, **sin** clonar ni resetear (no se
pisan cambios locales).

In [ ]:
def ejecutar(cmd, cwd=None, extra_env=None, check=True):
    """Corre un comando mostrando su salida; aborta si falla."""
    env = {**os.environ, **(extra_env or {})}
    print("$", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None,
                       env=env, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if check and r.returncode:
        raise RuntimeError("Fallo (%d): %s" % (r.returncode, " ".join(str(c) for c in cmd)))
    return r


if EN_COLAB:
    if (REPO_DIR / ".git").exists():
        ejecutar(["git", "-C", REPO_DIR, "fetch", "origin", RAMA])
        ejecutar(["git", "-C", REPO_DIR, "checkout", "-B", RAMA, "origin/" + RAMA])
        ejecutar(["git", "-C", REPO_DIR, "reset", "--hard", "origin/" + RAMA])
    else:
        ejecutar(["git", "clone", "--branch", RAMA, REPO_URL, REPO_DIR])
    print(ejecutar(["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).stdout.strip())
else:
    print("Modo local: se usa el repo en", REPO_DIR, "(no se clona ni se resetea).")

## 3 · Instalar dependencias

Cada pieza trae su `requirements.txt`; el retrieval no, así que sus dependencias
se instalan aparte.

In [ ]:
for req in ["M3/corpus", "M3/tools", "M3/generacion", "M3/ragas"]:
    ejecutar([sys.executable, "-m", "pip", "install", "-q", "-r", REPO_DIR / req / "requirements.txt"])

ejecutar([sys.executable, "-m", "pip", "install", "-q",
          "sentence-transformers", "chromadb", "rank_bm25", "groq", "python-dotenv", "PyYAML"])
print("Dependencias instaladas.")

## 4 · Datos y credenciales

- **Colab:** monta Drive y toma las claves de los Secretos.
- **Local:** toma las claves del entorno o de un `.env`.

En ambos casos se escribe un `.env` en la raíz del repo (ignorado por git) para
que los entry points lo lean.

In [ ]:
if EN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    for var in ("GROQ_API_KEY", "BIOPORTAL_API_KEY"):
        try:
            os.environ[var] = userdata.get(var)
        except Exception:
            pass
else:
    try:
        from dotenv import load_dotenv
        for p in [REPO_DIR / ".env", REPO_DIR / "M3/generacion/.env", REPO_DIR / "M2/harness/.env"]:
            if p.exists():
                load_dotenv(p, override=False)
    except Exception as e:
        print("Aviso cargando .env:", e)

lineas = ["PROJECT_ROOT=" + str(REPO_DIR)]
for var in ("GROQ_API_KEY", "BIOPORTAL_API_KEY"):
    if os.environ.get(var):
        lineas.append(var + "=" + os.environ[var])
(REPO_DIR / ".env").write_text("\n".join(lineas) + "\n", encoding="utf-8")
print("Claves presentes:", [v for v in ("GROQ_API_KEY", "BIOPORTAL_API_KEY") if os.environ.get(v)])
print(".env escrito en", REPO_DIR / ".env")

## 5 · Configuración del retrieval según el entorno

El YAML del repo (`config_retrieval.yaml`) apunta a Drive para Colab. En **local**
se arma un YAML equivalente (sin tocar el del repo) con el perfil `real` apuntando
a `DATA_ROOT`, y se exporta `M3_RETRIEVAL_CONFIG` para que los módulos lo usen.

In [ ]:
import yaml

os.environ["M3_REPO_ROOT"] = str(REPO_DIR)
os.environ["M3_DATA_ROOT"] = str(DATA_ROOT)

if EN_COLAB:
    print("Colab: se usa M3/retrieval/config_retrieval.yaml (rutas de Drive).")
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    cfg_r = yaml.safe_load((REPO_DIR / "M3/retrieval/config_retrieval.yaml").read_text(encoding="utf-8"))
    base = str(DATA_ROOT)
    cfg_r["rutas"]["real"].update({
        "corpus_json": base + "/data/guias_clinicas/chunks",
        "chroma_origen": base + "/data/chroma_guias",
        "chroma_dir": base + "/chroma_local",
        "manifest": base + "/data/guias_clinicas/corpus_manifest.json",
        "salida": "M3/retrieval/outputs",
    })
    cfg_r["rutas"]["mock"]["chroma_dir"] = base + "/chroma_mock"
    local_cfg = DATA_ROOT / "config_retrieval.local.yaml"
    local_cfg.write_text(yaml.safe_dump(cfg_r, allow_unicode=True, sort_keys=False), encoding="utf-8")
    os.environ["M3_RETRIEVAL_CONFIG"] = str(local_cfg)
    print("YAML de retrieval local:", local_cfg)

## 6 · Verificar recursos

Comprueba las guías (PDF), el resultado de M2 para el cruce y el eval set.

In [ ]:
guia_dir = DATA_ROOT / "data/guias_clinicas"
checks = {
    "Guías PDF (data/guias_clinicas)": guia_dir if guia_dir.exists() else (REPO_DIR / "M3/corpus/guias_clinicas"),
    "Resultado M2 (para el cruce)": REPO_DIR / "M2/ejecucion/outputs_gold/resultado_dimension1.json",
    "Eval set del equipo": REPO_DIR / EJEMPLOS,
}
for nombre, p in checks.items():
    print("%s: %s -- %s" % (nombre, "OK" if p.exists() else "FALTA", p))

ejemplos = REPO_DIR / EJEMPLOS
if ejemplos.exists():
    ENTIDADES = EJEMPLOS
elif (REPO_DIR / EJEMPLOS_ALT).exists():
    ENTIDADES = EJEMPLOS_ALT
    print("Aviso: no existe %s; se usa %s" % (EJEMPLOS, EJEMPLOS_ALT))
else:
    raise FileNotFoundError("No hay eval set de entidades.")

## 7 · Pruebas unitarias

Rápidas y sin descargar modelos. (Algunas pruebas de retrieval dependen del perfil
`corpus` activo en el YAML; pueden fallar si el YAML trae `real`.)

In [ ]:
ejecutar([sys.executable, "-m", "unittest", "test_retrieval"], cwd=REPO_DIR / "M3/retrieval", check=False)
ejecutar([sys.executable, "-m", "unittest", "test_generacion"], cwd=REPO_DIR / "M3/generacion", check=False)

## 8 · Corpus (Isa): ingesta de las guías + índice

Ingesta → chunking → embeddings → Chroma + manifest. Es **idempotente**: las
fuentes ya indexadas se saltan por caché de `doc_id`.

In [ ]:
if CORRER_CORPUS:
    import shutil
    destino = DATA_ROOT / "data/guias_clinicas"
    destino.mkdir(parents=True, exist_ok=True)
    for pdf in sorted((REPO_DIR / "M3/corpus/guias_clinicas").glob("*.pdf")):
        if not (destino / pdf.name).exists():
            shutil.copy2(pdf, destino / pdf.name)
            print("copiada:", pdf.name)
    ejecutar([sys.executable, "run_corpus.py", "--config", "config.yaml", "--fuentes", "fuentes.yaml"],
             cwd=REPO_DIR / "M3/corpus", extra_env={"PROJECT_ROOT": str(DATA_ROOT)})
else:
    print("CORRER_CORPUS=False: se usa el índice ya construido.")

## 9 · Retrieval (Pau): calibración de umbrales y delta

Experimento controlado (ingenuo vs. híbrido vs. reranker vs. sistema completo).
Escribe `umbrales.json` y `deltas_s08.csv` en `M3/retrieval/outputs/`.

In [ ]:
if CORRER_EXPERIMENTO:
    r = ejecutar([sys.executable, "experimento_s08.py", "--corpus", "real"],
                 cwd=REPO_DIR / "M3/retrieval", check=False)
    if r.returncode:
        print("\nAVISO: el experimento de calibracion fallo "
              "(las consultas del corpus real pueden estar sin etiquetar). "
              "Se continua con los umbrales provisionales del YAML.")
else:
    print("CORRER_EXPERIMENTO=False: se usan los umbrales provisionales del YAML.")


## 10 · Generación (Agustín): respuesta final del RAG

Encadena orquestación (cuándo se invoca la tool y qué hace si falla) + generación.
Consume el retrieval de Pau y la tool de Luis desde sus YAML y escribe
`M3/outputs/resultado_generacion.json`, que consume RAGAS.

In [ ]:
ejecutar([sys.executable, "run_generacion.py", "--config", "config.yaml", "--entidades", ENTIDADES],
         cwd=REPO_DIR / "M3/generacion", extra_env={"PROJECT_ROOT": str(REPO_DIR)})

## 11 · RAGAS + cruce con el harness de M2 (Luis)

Faithfulness, context precision, context recall y answer relevancy, cruzados con
el F1 de extracción de M2 para diagnosticar dónde falla el sistema.

In [ ]:
ejecutar([sys.executable, "run_ragas.py", "--config", "config.yaml", "--modo", MODO_RAGAS],
         cwd=REPO_DIR / "M3/ragas", extra_env={"PROJECT_ROOT": str(REPO_DIR)})

## 12 · Persistir los artefactos

Copia los resultados a `DATA_ROOT/resultados_pipeline` y arma un zip. En Colab
también ofrece la descarga.

In [ ]:
import shutil

persist = DATA_ROOT / "resultados_pipeline"
persist.mkdir(parents=True, exist_ok=True)
for origen, nombre in [(REPO_DIR / "M3/outputs", "generacion"),
                       (REPO_DIR / "M3/retrieval/outputs", "retrieval")]:
    if origen.exists():
        shutil.copytree(origen, persist / nombre, dirs_exist_ok=True)
        print("copiado:", persist / nombre)

zip_path = shutil.make_archive(str(DATA_ROOT / "resultados_M3"), "zip", str(persist))
print("zip:", zip_path)

if EN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("Descarga manual en:", zip_path, "|", e)